# Mechanistic consumption → signaling → PKA → sensor model

This notebook complements the empirical GLM in notebook 10. It uses the same complete-recording cache and fits a minimal dynamical cascade with a recoverable adaptation state.

The latent states are effective signaling states—not uniquely measured neuromodulator, cAMP, or PKA concentrations. Identifiability diagnostics below indicate whether the lifetime data can distinguish their rate constants.

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares

## Configuration

Run notebook 10's cache-building cell first. This notebook does not reread `.doric` files. `consumption` is defined as licking after the first Ensure delivery; all licking also enters a separate nuisance state, which no-Ensure recordings help constrain.

In [ ]:
GLM_OUTPUT_FOLDER = Path.cwd() / "analysis_output_adaptive_glm"
CACHE_FILE = GLM_OUTPUT_FOLDER / "full_recording_glm_cache.npz"
OUTPUT_FOLDER = Path.cwd() / "analysis_output_mechanistic_pka"
MOUSE_FILTER = []
DATE_FILTER = []
RECORDING_TEXT = ""
MODEL_INTERVAL_S = 0.2
ROBUST_LOSS = "soft_l1"
ROBUST_SCALE_NS = 0.001
MAX_FUNCTION_EVALUATIONS = 400
EQUAL_WEIGHT_RECORDINGS = True
if not CACHE_FILE.is_file():
    raise FileNotFoundError(f"Run notebook 10's full-recording cache cell first: {CACHE_FILE}")

In [ ]:
with np.load(CACHE_FILE, allow_pickle=False) as cache:
    cached_time = cache["time_s"]
    cached_delta_tau = cache["delta_tau_ns"]
    cached_licks = cache["lick_counts_per_bin"]
    cached_ensures = cache["ensure_counts_per_bin"]
    cache_interval_s = float(cache["bin_width_s"])
    all_mice = cache["mouse"].astype(str)
    all_dates = cache["session_date"].astype(str)
    all_recordings = cache["recording"].astype(str)
selected = np.ones(all_mice.size, dtype=bool)
if MOUSE_FILTER: selected &= np.isin(all_mice, np.asarray(MOUSE_FILTER, dtype=str))
if DATE_FILTER: selected &= np.isin(all_dates, np.asarray(DATE_FILTER, dtype=str))
if RECORDING_TEXT: selected &= np.char.find(np.char.lower(all_recordings), RECORDING_TEXT.casefold()) >= 0
factor = max(1, int(round(MODEL_INTERVAL_S / cache_interval_s)))
model_interval_s = factor * cache_interval_s
recordings = []
for source_index in np.flatnonzero(selected):
    valid_length = int(np.sum(np.isfinite(cached_time[source_index])))
    usable_length = (valid_length // factor) * factor
    if usable_length < factor * 10:
        continue
    def block_mean(values):
        return np.nanmean(values[:usable_length].reshape(-1, factor), axis=1)
    def block_sum(values):
        return np.nansum(values[:usable_length].reshape(-1, factor), axis=1)
    time = block_mean(cached_time[source_index])
    delta_tau = block_mean(cached_delta_tau[source_index])
    lick_counts = block_sum(cached_licks[source_index])
    ensure_counts = block_sum(cached_ensures[source_index])
    lick_rate = lick_counts / model_interval_s
    reward_available = np.cumsum(ensure_counts) > 0
    consumption_input = lick_rate * reward_available
    recordings.append({"mouse": all_mice[source_index], "date": all_dates[source_index], "recording": all_recordings[source_index], "time": time, "y": delta_tau, "lick_rate": lick_rate, "ensure_counts": ensure_counts, "consumption_input": consumption_input})
print(f"Prepared {len(recordings)} complete recordings at {model_interval_s:g}-s resolution")
print(f"No-Ensure: {sum(np.sum(item['ensure_counts']) == 0 for item in recordings)}; Ensure: {sum(np.sum(item['ensure_counts']) > 0 for item in recordings)}")

## Dynamical model

The consumption input drives an upstream state through the current adaptation availability. The upstream state drives a saturating PKA state, which drives the observed sensor state. A separate filtered-lick state captures lifetime variation associated with licking in both reward and no-reward recordings. Recording-specific intercept and linear drift terms are nuisance parameters solved analytically at every iteration.

In [ ]:
PARAMETER_NAMES = ["input_gain", "tau_upstream_s", "tau_pka_s", "tau_sensor_s", "adaptation_rate", "tau_recovery_s", "sensor_scale_ns", "tau_lick_s", "lick_scale_ns_per_hz"]
INITIAL_PARAMETERS = np.asarray([0.15, 2.0, 5.0, 1.0, 0.03, 90.0, -0.015, 1.0, 0.0001])
LOWER_BOUNDS = np.asarray([0.0001, 0.2, 0.2, 0.1, 0.0, 2.0, -0.2, 0.1, -0.02])
UPPER_BOUNDS = np.asarray([10.0, 60.0, 120.0, 30.0, 2.0, 1800.0, 0.2, 30.0, 0.02])

def exponential_update(value, target, tau_s, dt):
    return target + (value - target) * np.exp(-dt / tau_s)

def simulate_states(parameters, item):
    input_gain, tau_upstream, tau_pka, tau_sensor, adaptation_rate, tau_recovery, sensor_scale, tau_lick, lick_scale = parameters
    sample_count = item["time"].size
    upstream = np.zeros(sample_count); pka = np.zeros(sample_count); sensor = np.zeros(sample_count)
    adaptation = np.ones(sample_count); lick_state = np.zeros(sample_count)
    for index in range(1, sample_count):
        consumption = item["consumption_input"][index - 1]
        recovery_rate = 1.0 / tau_recovery
        total_adaptation_rate = recovery_rate + adaptation_rate * consumption
        adaptation_target = recovery_rate / total_adaptation_rate
        adaptation[index] = adaptation_target + (adaptation[index - 1] - adaptation_target) * np.exp(-total_adaptation_rate * model_interval_s)
        upstream_target = input_gain * consumption * adaptation[index]
        upstream[index] = exponential_update(upstream[index - 1], upstream_target, tau_upstream, model_interval_s)
        pka_target = upstream[index] / (1.0 + upstream[index])
        pka[index] = exponential_update(pka[index - 1], pka_target, tau_pka, model_interval_s)
        sensor[index] = exponential_update(sensor[index - 1], pka[index], tau_sensor, model_interval_s)
        lick_state[index] = exponential_update(lick_state[index - 1], item["lick_rate"][index - 1], tau_lick, model_interval_s)
    dynamic_prediction = sensor_scale * sensor + lick_scale * lick_state
    return dynamic_prediction, {"upstream": upstream, "pka": pka, "sensor": sensor, "adaptation": adaptation, "lick_state": lick_state}

def recording_prediction(parameters, item):
    dynamic_prediction, states = simulate_states(parameters, item)
    normalized_time = item["time"] / max(item["time"][-1], model_interval_s)
    nuisance_design = np.column_stack([np.ones(item["time"].size), normalized_time])
    nuisance_coefficients = np.linalg.lstsq(nuisance_design, item["y"] - dynamic_prediction, rcond=None)[0]
    prediction = dynamic_prediction + nuisance_design @ nuisance_coefficients
    return prediction, states, nuisance_coefficients

def residual_vector(parameters, selected_recordings=None):
    active = recordings if selected_recordings is None else selected_recordings
    residuals = []
    for item in active:
        prediction, _, _ = recording_prediction(parameters, item)
        residual = prediction - item["y"]
        if EQUAL_WEIGHT_RECORDINGS:
            residual = residual / np.sqrt(residual.size)
        residuals.append(residual)
    return np.concatenate(residuals)

In [ ]:
fit = least_squares(residual_vector, INITIAL_PARAMETERS, bounds=(LOWER_BOUNDS, UPPER_BOUNDS), loss=ROBUST_LOSS, f_scale=ROBUST_SCALE_NS, max_nfev=MAX_FUNCTION_EVALUATIONS, x_scale="jac", verbose=1)
parameters = fit.x
residuals = residual_vector(parameters)
degrees_of_freedom = max(1, residuals.size - parameters.size)
residual_variance = float(np.sum(residuals ** 2) / degrees_of_freedom)
covariance = np.linalg.pinv(fit.jac.T @ fit.jac) * residual_variance
standard_errors = np.sqrt(np.maximum(0.0, np.diag(covariance)))
singular_values = np.linalg.svd(fit.jac, compute_uv=False)
jacobian_condition = float(np.inf if singular_values[-1] == 0 else singular_values[0] / singular_values[-1])
parameter_table = []
for name, estimate, standard_error, lower, upper in zip(PARAMETER_NAMES, parameters, standard_errors, LOWER_BOUNDS, UPPER_BOUNDS):
    near_bound = bool((estimate - lower) < 0.01 * (upper - lower) or (upper - estimate) < 0.01 * (upper - lower))
    parameter_table.append({"parameter": name, "estimate": estimate, "approx_standard_error": standard_error, "near_bound": near_bound})
print(f"Converged: {fit.success} ({fit.message})")
print(f"Jacobian condition number: {jacobian_condition:.3g}")
for row in parameter_table: print(row)
if jacobian_condition > 1e6 or any(row["near_bound"] for row in parameter_table):
    print("WARNING: Some mechanistic parameters are weakly identifiable. Interpret latent trajectories and predictions more strongly than individual rate constants.")

In [ ]:
ensure_recording_indices = [index for index, item in enumerate(recordings) if np.sum(item["ensure_counts"]) > 0]
if not ensure_recording_indices:
    raise ValueError("At least one Ensure-containing recording is required for the mechanistic cascade plot")
example_index = max(ensure_recording_indices, key=lambda index: np.sum(recordings[index]["ensure_counts"]))
example = recordings[example_index]
example_prediction, example_states, nuisance = recording_prediction(parameters, example)
ensure_times = example["time"][example["ensure_counts"] > 0]
figure, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True, constrained_layout=True)
figure.suptitle(f"Mechanistic cascade: {example['mouse']} {example['recording']}")
axes[0].plot(example["time"], example["y"], color="0.7", linewidth=0.8, label="Observed lifetime")
axes[0].plot(example["time"], example_prediction, color="tab:blue", linewidth=1.5, label="Mechanistic prediction")
axes[0].set_ylabel("Delta tau (ns)"); axes[0].legend()
axes[1].plot(example["time"], example_states["upstream"], label="Upstream signaling")
axes[1].plot(example["time"], example_states["pka"], label="Latent PKA")
axes[1].plot(example["time"], example_states["sensor"], label="Sensor state")
axes[1].set_ylabel("Normalized state"); axes[1].legend()
axes[2].plot(example["time"], example_states["adaptation"], color="tab:purple")
axes[2].set_ylabel("Availability"); axes[2].set_ylim(-0.05, 1.05); axes[2].set_title("Adaptation state (1 = fully available)")
axes[3].plot(example["time"], example["lick_rate"], color="tab:green", linewidth=0.8, label="Lick rate")
axes[3].vlines(ensure_times, 0, max(1.0, np.nanmax(example["lick_rate"])), color="tab:orange", alpha=0.45, linewidth=0.8, label="Ensure TTL")
axes[3].set_ylabel("Lick rate (Hz)"); axes[3].set_xlabel("Recording time (s)"); axes[3].legend()
plt.show()

In [ ]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
figure.savefig(OUTPUT_FOLDER / "mechanistic_pka_cascade.png", dpi=200)
with (OUTPUT_FOLDER / "mechanistic_parameters.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(parameter_table[0])); writer.writeheader(); writer.writerows(parameter_table)
np.savez_compressed(OUTPUT_FOLDER / "mechanistic_fit_results.npz", parameter_names=np.asarray(PARAMETER_NAMES), parameter_estimates=parameters, approximate_standard_errors=standard_errors, jacobian_condition=np.asarray(jacobian_condition), example_time_s=example["time"], example_observed_delta_tau_ns=example["y"], example_predicted_delta_tau_ns=example_prediction, example_upstream=example_states["upstream"], example_pka=example_states["pka"], example_sensor=example_states["sensor"], example_adaptation=example_states["adaptation"])
print(f"Saved mechanistic-model outputs to {OUTPUT_FOLDER.resolve()}")

## Interpretation limits

A good fit does not prove that the intermediate states correspond uniquely to neuromodulator, cAMP, and PKA. Linear cascades can collapse to nearly the same composite kernel, and the regular 15-second delivery schedule limits separation of slow kinetics from response overlap. Strong claims about individual biochemical time constants require independent calibration, perturbation, or an additional sensor.

Use notebook 10's GLM as the empirical benchmark. Prefer this mechanistic model only when it predicts held-out recordings comparably well and its parameter estimates remain stable across initial values and reasonable model variants.